In [1]:
import numpy as np
from scipy import stats
from math import ceil
from pathlib import Path
import matplotlib.pyplot as plt
from contact_plots import get_data_from_files, get_cdf_of_data, do_chi_square_test
import warnings
warnings.filterwarnings('ignore')

In [2]:
data_folder = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Simulations\2021_12\transfer\results_20211215')

contact_data_list, infra_and_measure_name_2_data = get_data_from_files(data_folder)

In [3]:
data_names_small = (
    '1.0_customer_customer_weight_over_contacts',
    '1.0_customer_customer_weight_over_agents',
    '1.0_customer_customer_contacts_over_agents',
    '1.0_staff_customer_weight_over_contacts',
    '1.0_staff_customer_weight_over_agents',
    '1.0_staff_customer_contacts_over_agents',
    None,
    '1.0_customer_staff_weight_over_agents',
    '1.0_customer_staff_contacts_over_agents',
)

data_names_middle = (
    '1.5_customer_customer_weight_over_contacts',
    '1.5_customer_customer_weight_over_agents',
    '1.5_customer_customer_contacts_over_agents',
    '1.5_staff_customer_weight_over_contacts',
    '1.5_staff_customer_weight_over_agents',
    '1.5_staff_customer_contacts_over_agents',
    None,
    '1.5_customer_staff_weight_over_agents',
    '1.5_customer_staff_contacts_over_agents',
)

data_names_large = (
    '2.0_customer_customer_weight_over_contacts',    
    '2.0_customer_customer_weight_over_agents',
    '2.0_customer_customer_contacts_over_agents',
    '2.0_staff_customer_weight_over_contacts',
    '2.0_staff_customer_weight_over_agents',
    '2.0_staff_customer_contacts_over_agents',
    None,
    '2.0_customer_staff_weight_over_agents',
    '2.0_customer_staff_contacts_over_agents',
)

data_name_groups = {data_names_middle:'1.5',
                    data_names_small:'1.0',                      
                    data_names_large:'2.0'}

data_name_groups_inv = {value:key for key, value in data_name_groups.items()}

max_string_length = 0
for data_names in data_name_groups.keys():
    for data_name in data_names:
        if data_name is not None and len(data_name) > max_string_length:
            max_string_length = len(data_name)

In [4]:
line_styles = {
    'base_case':'-',
    'busy_corner':'--',
    'long_restaurant':':'    
}

colors = {
    'nn_nts_bvd':'r',
    'n_nts_bvd':'g',
    'nn_ts_bvd':'b',
    'nn_nts_svd':'y',
    'nn_nts_lvd':'c'
}
    

## Results for 1.5m distance

In [5]:
%matplotlib wx

col_count = 3
row_count = int(ceil(len(data_names_small)/col_count))
fig, axes = plt.subplots(row_count, col_count)
fig.suptitle('Overall', fontsize=16)
fig.canvas.manager.set_window_title('Overall')
axes = axes.flatten()

lines = []
labels = []
for ax_nr, data_name in enumerate(data_names_middle):
    ax = axes[ax_nr]
    if data_name is None:
        ax.legend(lines, labels)
        continue    
    ax.grid()
    ax.set_title(data_name)
    for infra_name, line_style in line_styles.items():
        for measure_name, color in colors.items():
            contact_data = infra_and_measure_name_2_data[infra_name][measure_name][data_name]
            x, y = get_cdf_of_data(contact_data)
            line = ax.plot(x,y, label=f'{infra_name} - {measure_name}', linestyle=line_style, color=color)
            if ax_nr == 0:
                lines.append(line[0]) 
                labels.append(f'{infra_name} - {measure_name}')
        
for infra_name in line_styles:
    print(f'\n======== {infra_name} ========\n')
    
    col_count = 3
    row_count = int(ceil(len(data_names_small)/col_count))
    fig, axes = plt.subplots(row_count, col_count)
    axes = axes.flatten()
    fig.suptitle(infra_name, fontsize=16)
    fig.canvas.manager.set_window_title(infra_name)

    lines = []
    labels = []
    legend_ax = None
    for ax_nr, data_name in enumerate(data_names_middle):
        print(f'----- {data_name} -----')
        ax = axes[ax_nr]
        if data_name is None:
            legend_ax = ax            
            continue    
        ax.grid()
        ax.set_title(data_name)
        diff_stats = {}
        for measure_name, color in colors.items():
            contact_data = infra_and_measure_name_2_data[infra_name][measure_name][data_name]
            x, y = get_cdf_of_data(contact_data)
            line = ax.plot(x,y, label=f'{measure_name}', color=color)
            
            if ax_nr == 0:
                lines.append(line[0]) 
                labels.append(f'{measure_name}')

            diff_stats[measure_name] = {}
            for other_measure_name in colors:
                if other_measure_name == measure_name:
                    diff_stats[measure_name][other_measure_name] = None
                elif other_measure_name in diff_stats and measure_name in diff_stats[other_measure_name]:
                    diff_stats[measure_name][other_measure_name] = None
                else:
                    other_contact_data = infra_and_measure_name_2_data[infra_name][other_measure_name][data_name]
                    if 'contacts_over_agents' in data_name:
                        base_array = contact_data
                        compare_array = other_contact_data
                        base_values, base_counts_org = np.unique(base_array, return_counts=True)
                        base_counts = base_counts_org
                        base_counts_dict = dict(zip(base_values, base_counts))
                        compare_values, compare_counts_org = np.unique(compare_array, return_counts=True)
                        compare_counts = compare_counts_org/sum(compare_counts_org)*sum(base_counts_org)
                        compare_counts_dict = dict(zip(compare_values, compare_counts))
                        values = np.unique(np.concatenate((base_values, compare_values))) 
                        base_counts = [base_counts_dict[value] if value in base_counts_dict else 0 for value in values]
                        compare_counts = [compare_counts_dict[value] if value in compare_counts_dict else 0 for value in values]
                        
                        if len(values) == 1:
                            p_value = 1.0
                        else:
                            res = stats.power_divergence(compare_counts, base_counts)
                            p_value = res.pvalue                    
                    else:
                        res = stats.cramervonmises_2samp(contact_data, other_contact_data)
                        p_value = res.pvalue
                    
                    main_mean = np.mean(contact_data)
                    other_mean = np.mean(other_contact_data)
                    print(f'{measure_name:>10s} - {other_measure_name:>10s} = {p_value:.5f},   {(1 - main_mean/other_mean)*100:6.2f}%   ({main_mean:.2f}, {other_mean:.2f}),    ({len(contact_data)} - {len(other_contact_data)})')
                    diff_stats[measure_name][other_measure_name] = p_value
                    
                    
    legend_ax.legend(lines, labels)
    

            
plt.show()



======== base_case ========

----- 1.5_customer_customer_weight_over_contacts -----
nn_nts_bvd -  n_nts_bvd = 0.92910,    -0.31%   (2.81, 2.80),    (63572 - 63663)
nn_nts_bvd -  nn_ts_bvd = 0.00000,     8.52%   (2.81, 3.07),    (63572 - 52471)
nn_nts_bvd - nn_nts_svd = 0.00001,    -3.53%   (2.81, 2.72),    (63572 - 94349)
nn_nts_bvd - nn_nts_lvd = 0.00000,     1.95%   (2.81, 2.87),    (63572 - 33660)
 n_nts_bvd -  nn_ts_bvd = 0.00000,     8.80%   (2.80, 3.07),    (63663 - 52471)
 n_nts_bvd - nn_nts_svd = 0.00002,    -3.20%   (2.80, 2.72),    (63663 - 94349)
 n_nts_bvd - nn_nts_lvd = 0.00000,     2.25%   (2.80, 2.87),    (63663 - 33660)
 nn_ts_bvd - nn_nts_svd = 0.00000,   -13.16%   (3.07, 2.72),    (52471 - 94349)
 nn_ts_bvd - nn_nts_lvd = 0.00000,    -7.18%   (3.07, 2.87),    (52471 - 33660)
nn_nts_svd - nn_nts_lvd = 0.00000,     5.29%   (2.72, 2.87),    (94349 - 33660)
----- 1.5_customer_customer_weight_over_agents -----
nn_nts_bvd -  n_nts_bvd = 0.99054,    -0.16%   (32.00, 31.95),

In [ ]:
for measure_name in colors:
    print(f'\n======== {measure_name} ========\n')
    
    col_count = 3
    row_count = int(ceil(len(data_names_small)/col_count))
    fig, axes = plt.subplots(row_count, col_count)
    axes = axes.flatten()
    fig.suptitle(measure_name, fontsize=16)
    fig.canvas.manager.set_window_title(measure_name)

    lines = []
    labels = []
    for ax_nr, data_name in enumerate(data_names_middle):
        print(f'----- {data_name} -----')
        ax = axes[ax_nr]
        if data_name is None:
            ax.legend(lines, labels)
            continue    
        ax.grid()
        ax.set_title(data_name)
        diff_stats = {}
        for infra_name, line_style in line_styles.items():
            contact_data = infra_and_measure_name_2_data[infra_name][measure_name][data_name]
            x, y = get_cdf_of_data(contact_data)
            line = ax.plot(x,y, label=f'{infra_name}', linestyle=line_style)
            if ax_nr == 0:
                lines.append(line[0]) 
                labels.append(f'{infra_name}')

            diff_stats[infra_name] = {}
            for other_infra_name in line_styles:
                if other_infra_name == infra_name:
                    diff_stats[infra_name][other_infra_name] = None
                elif other_infra_name in diff_stats and infra_name in diff_stats[other_infra_name]:
                    diff_stats[infra_name][other_infra_name] = None
                else:
                    other_contact_data = infra_and_measure_name_2_data[other_infra_name][measure_name][data_name]
                    if 'contacts_over_agents' in data_name:
                        base_array = contact_data
                        compare_array = other_contact_data
                        base_values, base_counts_org = np.unique(base_array, return_counts=True)
                        base_counts = base_counts_org
                        base_counts_dict = dict(zip(base_values, base_counts))
                        compare_values, compare_counts_org = np.unique(compare_array, return_counts=True)
                        compare_counts = compare_counts_org/sum(compare_counts_org)*sum(base_counts_org)
                        compare_counts_dict = dict(zip(compare_values, compare_counts))
                        values = np.unique(np.concatenate((base_values, compare_values))) 
                        base_counts = [base_counts_dict[value] if value in base_counts_dict else 0 for value in values]
                        compare_counts = [compare_counts_dict[value] if value in compare_counts_dict else 0 for value in values]
                        
                        if len(values) == 1:
                            p_value = 1.0
                        else:
                            res = stats.power_divergence(compare_counts, base_counts)
                            p_value = res.pvalue
                    else:
                        res = stats.cramervonmises_2samp(contact_data, other_contact_data)
                        p_value = res.pvalue
                    
                    main_mean = np.mean(contact_data)
                    other_mean = np.mean(other_contact_data)
                    print(f'{infra_name:>15s} - {other_infra_name:>15s} = {p_value:.5f},   {(1 - main_mean/other_mean)*100:6.2f}%   ({main_mean:.2f}, {other_mean:.2f}),    ({len(contact_data)} - {len(other_contact_data)})')
                    diff_stats[infra_name][other_infra_name] = p_value

In [ ]:
print(contact_data.keys())
print(contact_data['base_case_nn_nts_bvd'].keys())

## Relative differences

In [ ]:
import numpy as np
from pathlib import Path
from contact_plots import get_data_from_files 

data_folder = Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Simulations\2021_12\transfer\results_20211213')

contact_data = get_data_from_files(data_folder)

combined_data_filenames = {'Base':Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Scenarios\paper_scenarios\trb_paper_base_combined.stats.npz'),
                          'No_shift':Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Scenarios\paper_scenarios\trb_paper_no_shifts_combined.stats.npz'),
                          'Longer':Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Scenarios\paper_scenarios\trb_paper_longer_visit_combined.stats.npz'),
                          'No_neighbourhood':Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Scenarios\paper_scenarios\trb_paper_no_neighbourhoods_combined.stats.npz'),
                          'More_toilets':Path(r'C:\Users\martijnsparnaa\surfdrive\PhD\Activity scheduler\Scenarios\paper_scenarios\trb_paper_more_toilet_combined.stats.npz')
                          }

combined_data = {label:np.load(combined_data_filenames[label]) for label, filename in combined_data_filenames.items()} 
name_2_label = {'Base':'Base',
                'No_shift':'No time slots',
                'Longer':'Longer visits',
                'No_neighbourhood':'No neighbourhoods',
                'More_toilets':'More toilet visits'}


In [ ]:
data_names_small = (
    '1.0_customer_customer_weight_over_contacts',
    '1.0_customer_customer_weight_over_agents',
    '1.0_customer_customer_contacts_over_agents',
    '1.0_staff_customer_weight_over_contacts',
    '1.0_staff_customer_weight_over_agents',
    '1.0_staff_customer_contacts_over_agents',
    None,
    '1.0_customer_staff_weight_over_agents',
    '1.0_customer_staff_contacts_over_agents',
)

data_names_middle = (
    '1.5_customer_customer_weight_over_contacts',
    '1.5_customer_customer_weight_over_agents',
    '1.5_customer_customer_contacts_over_agents',
    '1.5_staff_customer_weight_over_contacts',
    '1.5_staff_customer_weight_over_agents',
    '1.5_staff_customer_contacts_over_agents',
    None,
    '1.5_customer_staff_weight_over_agents',
    '1.5_customer_staff_contacts_over_agents',
)

data_names_large = (
    '2.0_customer_customer_weight_over_contacts',    
    '2.0_customer_customer_weight_over_agents',
    '2.0_customer_customer_contacts_over_agents',
    '2.0_staff_customer_weight_over_contacts',
    '2.0_staff_customer_weight_over_agents',
    '2.0_staff_customer_contacts_over_agents',
    None,
    '2.0_customer_staff_weight_over_agents',
    '2.0_customer_staff_contacts_over_agents',
)

data_name_groups = {data_names_middle:'1.5',
                    data_names_small:'1.0',                      
                    data_names_large:'2.0'}

data_name_groups_inv = {value:key for key, value in data_name_groups.items()}

max_string_length = 0
for data_names in data_name_groups.keys():
    for data_name in data_names:
        if data_name is not None and len(data_name) > max_string_length:
            max_string_length = len(data_name)

In [ ]:
from contact_plots import get_cdf_of_data

weigths_middle_cdf_data = {}
weigths_sc_middle_cdf_data = {}
weigths_cc_middle_cdf_data = {}

dist_label = '1.5'
data_names = data_name_groups_inv[dist_label]
for label, data in combined_data.items():
    comb_data = None
    for data_name in data_names:
        if data_name is None or 'weight_over_contacts' not in data_name:
            continue
        if len(data[data_name]) == 0:
            continue
        if comb_data is None:
            comb_data = data[data_name]
        else:
            comb_data = np.concatenate((comb_data, data[data_name]))
        if 'staff_customer' in data_name:
            x, y = get_cdf_of_data(data[data_name]) 
            weigths_sc_middle_cdf_data[name_2_label[label]] = {'x':x, 'y':y}
        elif 'customer_customer' in data_name:
            x, y = get_cdf_of_data(data[data_name]) 
            weigths_cc_middle_cdf_data[name_2_label[label]] = {'x':x, 'y':y}
            
    x, y = get_cdf_of_data(comb_data)  
    weigths_middle_cdf_data[name_2_label[label]] = {'x':x, 'y':y}
    

In [ ]:
%matplotlib wx

from contact_plots import create_cfd_weights_paper_plot



create_cfd_weights_paper_plot(weigths_middle_cdf_data, weigths_cc_middle_cdf_data, weigths_sc_middle_cdf_data, figure_props={'figsize':(6.5,5)})


In [ ]:
from contact_plots import get_cdf_of_data

contacts_middle_cdf_data = {}

data_names = (
    '1.5_customer_customer_contacts_over_agents',
    '1.5_customer_staff_contacts_over_agents',
    '1.5_staff_customer_contacts_over_agents',    
)
titles = (
    '(a) Customer - Customer',
    '(b) Staff - Customer (Customer)',
    '(c) Staff - Customer (Staff)'
)

for label, data in combined_data.items():
    ax_ind = 0
    for data_name in data_names:
        if data_name is None or 'contacts_over_agents' not in data_name:
            continue
        if len(data[data_name]) == 0:
            continue
        x, y = get_cdf_of_data(data[data_name])        
        if titles[ax_ind] not in contacts_middle_cdf_data:
            contacts_middle_cdf_data[titles[ax_ind]] = {}
        contacts_middle_cdf_data[titles[ax_ind]][name_2_label[label]] = {'x':x, 'y':y}        
        ax_ind += 1
 

In [ ]:
%matplotlib wx

from contact_plots import create_cfd_contacts_paper_plot

create_cfd_contacts_paper_plot(contacts_middle_cdf_data, figure_props={'figsize':(6.5,5)})
